The following code snippets are copied from:
https://huggingface.co/blog/how-to-generate

# Load Model

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

torch_device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# add the EOS token as PAD token to avoid warnings
model = AutoModelForCausalLM.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id).to(torch_device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]



---


# Greedy Decoding

In [16]:
# encode context the generation is conditioned on
model_inputs = tokenizer('I enjoy walking with my cute dog', return_tensors='pt').to(torch_device)

# generate 40 new tokens
greedy_output = model.generate(**model_inputs, max_new_tokens=40)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with my dog. I'm not sure if I'll ever be able to walk with my dog.

I'm not sure




---


# Beam Search

In [30]:
# activate beam search and early_stopping
beam_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    num_beams=5,
    early_stopping=True
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I'm not sure if I'll ever be able to walk with him again. I'm not sure


In [31]:
# set no_repeat_ngram_size to 2
beam_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    num_beams=5,
    no_repeat_ngram_size=2,
    early_stopping=True
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I've been thinking about this for a while now, and I think it's time for me to


In [32]:
# set return_num_sequences > 1
beam_outputs = model.generate(
    **model_inputs,
    max_new_tokens=40,
    num_beams=5,
    no_repeat_ngram_size=2,
    num_return_sequences=5,
    early_stopping=True
)

# now we have 5 output sequences
print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I've been thinking about this for a while now, and I think it's time for me to
1: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with her again.

I've been thinking about this for a while now, and I think it's time for me to
2: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I've been thinking about this for a while now, and I think it's a good idea to
3: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I've been thinking about this for a while now, and I think it's time to take a
4: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I've been thinking about this for a while now, and I think it's a good idea



---


# Sampling

In [ ]:
# set seed to reproduce results. Feel free to change the seed though to get different results
from transformers import set_seed
set_seed(42)

# activate sampling and deactivate top_k by setting top_k sampling to 0
sample_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    do_sample=True,
    top_k=0
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

In [ ]:
# set seed to reproduce results. Feel free to change the seed though to get different results
set_seed(42)

# use temperature to decrease the sensitivity to low probability candidates
sample_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    do_sample=True,
    top_k=0,
    temperature=0.6,
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))



---


# Top-K Sampling

In [ ]:
# set seed to reproduce results. Feel free to change the seed though to get different results
set_seed(42)

# set top_k to 50
sample_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    do_sample=True,
    top_k=50
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))



---


# Top-p (nucleus) sampling

In [ ]:
# set seed to reproduce results. Feel free to change the seed though to get different results
set_seed(42)

# set top_k to 50
sample_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    do_sample=True,
    top_p=0.92,
    top_k=0
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

In [ ]:
# set seed to reproduce results. Feel free to change the seed though to get different results
set_seed(42)

# set top_k = 50 and set top_p = 0.95 and num_return_sequences = 3
sample_outputs = model.generate(
    **model_inputs,
    max_new_tokens=40,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    num_return_sequences=3,
)

print("Output:\n" + 100 * '-')
for i, sample_output in enumerate(sample_outputs):
  print("{}: {}".format(i, tokenizer.decode(sample_output, skip_special_tokens=True)))



---


# Softmax and Temperature

Reminder:

- $Softmax(z_i) = \frac{e^{z_i}} {∑^K_{j=1} e^{z_j} }$

In [34]:
import math

def softmax(logits):
  sMax = []
  sum = 0
  for j in logits:
    sum += math.exp(j) #calculating the term in the denominator -
  for i in logits:
    sMax.append(math.exp(i)/sum) #calculating the softmax for every value in logits
  return sMax

If simply a probability is required, why the formula above and not this one:

- $Softmax(z_i) = \frac{{z_i}} {∑^K_{j=1} {z_j} }$

???

In [35]:
def softmaxNoExp(logits):
  sMax = []
  sum = 0
  for j in logits:
    sum += j #calculating the term in the denominator -
  for i in logits:
    sMax.append(i/sum) #calculating the softmax for every value in logits
  return sMax

In [36]:
print (softmax([5,1,3,7,4]))
print (sum(softmax([5,1,3,7,4])))

print (softmaxNoExp([5,1,3,7,4]))
print (sum(softmaxNoExp([5,1,3,7,4])))

[0.11222605877167183, 0.0020554919663678, 0.015188145450392949, 0.8292446440257712, 0.041285659785796076]
0.9999999999999999
[0.25, 0.05, 0.15, 0.35, 0.2]
1.0


Properties of both:
- mapping unbound input values into a range between 0 and 1
- sums to 1
- highest input value will also be the highest value in the output

In [44]:
import torch.nn as nn
exampleLogits = torch.Tensor([5,1,3,7,4])

print (nn.functional.softmax(exampleLogits, dim=-1))

tensor([0.1122, 0.0021, 0.0152, 0.8292, 0.0413])




---


Real world model output (logits):

In [38]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load pre-trained model tokenizer and model
torch_device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(torch_device)

# Prepare input
text = "Hello, world!"
encoded_input = tokenizer(text, return_tensors='pt').to(torch_device)

# Get model output
with torch.no_grad():
    output = model(**encoded_input)

# The logits are usually in the `logits` attribute of the model output
logits = output.logits

print("Logits shape (batch_size, sequence_length, vocab_size):")
print(logits.shape)
print("\nLogits for the last token of the sequence (first 10 vocabulary items):")
print(logits[0, -1, :10])

print(logits[0, -1, :])

print(f"Maximum Logit Value: {logits[0, -1,torch.argmax(logits[0, -1, :])]}")
print(f"Minimum Logit Value: {logits[0, -1,torch.argmin(logits[0, -1, :])]}")
print(f"Average Logit Value: {torch.mean(logits[0, -1, :])}")


Logits shape (batch_size, sequence_length, vocab_size):
torch.Size([1, 4, 50257])

Logits for the last token of the sequence (first 10 vocabulary items):
tensor([-118.9063, -119.5879, -121.2070, -122.0629, -122.7480, -122.6188,
        -120.5190, -119.9568, -120.8519, -119.3643])
tensor([-118.9063, -119.5879, -121.2070,  ..., -130.6019, -128.5587,
        -116.1679])
Maximum Logit Value: -112.93436431884766
Minimum Logit Value: -141.03634643554688
Average Logit Value: -126.98442077636719


In [39]:
logitsReal =[-118.9063, -119.5879, -121.2070, -122.0629, -122.7480]
print (softmax(logitsReal))
print (softmaxNoExp(logitsReal))

[0.5987941240504565, 0.3028742914204185, 0.05999239712575287, 0.02549080603800661, 0.012848381365365443]
[0.19669796518547766, 0.19782548604072606, 0.20050384433992302, 0.20191969689275036, 0.20305300754112282]




---
Softmax with Temperature **T**:

- $Softmax(z_i) = \frac{e^{β z_i}} {∑^K_{j=1} e^{\beta z_j} }$
- with $β = \frac {1} {T}$

In [46]:
def softmaxWTemp(logits,temp):
  sMax = []
  sum = 0
  for j in logits:
    sum += math.exp(j/temp) #calculating the term in the denominator -
  for i in logits:
    sMax.append(math.exp(i/temp)/sum) #calculating the softmax for every value in logits
  return sMax

In [54]:
highTemp = 1000000
lowTemp = 0.01

print(softmaxWTemp(exampleLogits,highTemp))
print(softmaxWTemp(exampleLogits,lowTemp))

[0.2000001999996722, 0.19999940000049699, 0.1999997999997187, 0.20000060000051667, 0.19999999999959547]
[1.3838965267367376e-87, 2.650396553004311e-261, 1.9151695967140057e-174, 1.0, 5.148200222412014e-131]


In [55]:
print(nn.functional.softmax(exampleLogits/highTemp, dim=-1))

print(nn.functional.softmax(exampleLogits/lowTemp, dim=-1))

tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000])
tensor([0., 0., 0., 1., 0.])


**Questions:**

- Will (vanilla) sampling in combination with a very low (<<0.0001) Temperature lead to the same result as greedy decoding for a given time step?

- Will Top-p sampling with p = 1.0 lead to different results than the plain
sampling method?



---


# Example Generation function

https://github.com/karpathy/minGPT/blob/master/mingpt/model.py

In [ ]:
def generate(self, idx, max_new_tokens, temperature=1.0, do_sample=False, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # either sample from the distribution or take the most likely element
            if do_sample:
                idx_next = torch.multinomial(probs, num_samples=1)
            else:
                _, idx_next = torch.topk(probs, k=1, dim=-1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

- What is the main issue with auto-regressive decoding?